In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# This code reads the final enhanced frames and detects
# raveling regions on the road surface.
#
# FIXES APPLIED:
#   - Added Region of Interest (ROI) to ignore grass, dirt
#     edges, and sky areas at the frame borders
#   - Changed from Otsu to fixed threshold (value=30)
#     to avoid detecting all bright regions
#   - Smaller top-hat kernel (9x9 instead of 15x15)
#     to be less sensitive to large non-raveling patches
#   These three changes together reduce false detections
#   on grass, road edges, and other damage types.


# Step 1: Set Up Folders

input_folder  = "Final_Enhanced/"         # Final enhanced frames (Member 4)
gt_folder     = "GroundTruth_Raveling/"   # Your manually drawn masks (15 frames)
output_folder = "Member4_Segmentation/"   # All outputs saved here

os.makedirs(output_folder, exist_ok=True)

# Step 2: Read Raveling Frame Names

raveling_input = "Frames_Raveling/"

all_files = os.listdir(raveling_input)

frame_files = []
for f in all_files:
    if f.endswith(".jpg"):
        frame_files.append(f)

frame_files.sort(key=lambda x: int(x.split("_")[1].split(".")[0]))

print("Total raveling frames found:", len(frame_files))


# Step 3: Check Which Frames Have Ground Truth Masks

frames_with_gt = []
for f in frame_files:
    gt_path = gt_folder + f
    if os.path.exists(gt_path):
        frames_with_gt.append(f)

print("Frames with ground truth masks:", len(frames_with_gt))

if len(frames_with_gt) == 0:
    print("WARNING: No ground truth masks found in", gt_folder)

# Step 4: Segmentation Function
# ============================================================
# This function is used by both the processing loop and
# the metrics loop to make sure the same steps are applied
# consistently every time.

def segment_raveling_frame(img):

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ---- Region of Interest (ROI) ----
    # We only look at the central road area.
    # The left 12% and right 20% of the frame are cut off
    # because those areas contain grass and dirt edges
    # that look similar to raveling texture but are not.
    # The top 15% is cut off to remove sky and background.
    roi_mask  = np.zeros((h, w), dtype=np.uint8)
    left_cut  = int(w * 0.12)
    right_cut = int(w * 0.80)
    top_cut   = int(h * 0.15)
    cv2.rectangle(roi_mask, (left_cut, top_cut), (right_cut, h), 255, -1)

    # Apply ROI to grayscale so we only process the road area
    gray_roi = cv2.bitwise_and(gray, gray, mask=roi_mask)

    # ---- Top-Hat Operation ----
    # Top-hat highlights small bright texture features.
    # We use a smaller 9x9 kernel (previously 15x15).
    # A smaller kernel is less sensitive to large bright
    # patches like the sandy road edge or potholes.
    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    tophat = cv2.morphologyEx(gray_roi, cv2.MORPH_TOPHAT, kernel_tophat)

    # ---- Fixed Threshold ----
    # We use a fixed threshold of 30 instead of Otsu.
    # Otsu automatically picks the best threshold but it
    # was being too aggressive and detecting everything bright.
    # A fixed value of 30 only picks up the strongest
    # bright texture spots which are more likely to be raveling.
    # If too much is detected: increase this value (e.g. 40, 50)
    # If too little is detected: decrease this value (e.g. 20, 15)
    THRESHOLD_VALUE = 30
    _, binary = cv2.threshold(tophat, THRESHOLD_VALUE, 255, cv2.THRESH_BINARY)

    # Apply ROI again to make sure no edge leakage remained
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    # ---- Morphological Closing ----
    # Connect nearby raveling spots into regions
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close)

    # ---- Morphological Opening ----
    # Remove remaining small isolated noise spots
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    final_mask = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)

    return final_mask, gray, tophat, binary


# Step 5: Test on First Frame - Check the Output Visually

print("\nTesting on first frame...")

test_name = frame_files[0]
test_img  = cv2.imread(raveling_input + test_name)

if test_img is None:
    print("Could not read the first frame. Check the folder path.")
else:

    final_mask, gray, tophat, binary = segment_raveling_frame(test_img)

    # Show all processing steps clearly
    plt.figure(figsize=(15, 3))

    plt.subplot(1, 5, 1)
    plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    plt.title("Original Frame")
    plt.axis('off')

    plt.subplot(1, 5, 2)
    plt.imshow(gray, cmap='gray')
    plt.title("Grayscale")
    plt.axis('off')

    plt.subplot(1, 5, 3)
    plt.imshow(tophat, cmap='gray')
    plt.title("After Top-Hat (9x9)")
    plt.axis('off')

    plt.subplot(1, 5, 4)
    plt.imshow(binary, cmap='gray')
    plt.title("After Threshold (value=30)")
    plt.axis('off')

    plt.subplot(1, 5, 5)
    plt.imshow(final_mask, cmap='gray')
    plt.title("Final Mask (with ROI)")
    plt.axis('off')

    plt.suptitle("Raveling Segmentation Steps - Fixed Version (" + test_name + ")")
    plt.tight_layout()
    plt.savefig(output_folder + "test_steps_" + test_name)
    plt.show()

    print("Test done. Check:", output_folder + "test_steps_" + test_name)
    print()
    print("If you see TOO MUCH detected (noisy): open the code and")
    print("change THRESHOLD_VALUE = 30  to  THRESHOLD_VALUE = 40 or 50")
    print()
    print("If you see TOO LITTLE detected (almost empty): change")
    print("THRESHOLD_VALUE = 30  to  THRESHOLD_VALUE = 20 or 15")


# Step 6: Process All Raveling Frames

print("\nProcessing all raveling frames...")

for frame_name in frame_files:

    img = cv2.imread(raveling_input + frame_name)

    if img is None:
        print("Could not read:", frame_name, "- Skipping.")
        continue

    final_mask, gray, tophat, binary = segment_raveling_frame(img)

    # ---- Find Contours ----
    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # ---- Filter by Area ----
    # Only keep regions larger than 200 pixels
    large_contours = []
    for c in contours:
        if cv2.contourArea(c) > 200:
            large_contours.append(c)

    # ---- Draw Contours on Original Frame ----
    marked = img.copy()
    cv2.drawContours(marked, large_contours, -1, (255, 0, 255), 2)
    # Purple/magenta color (255, 0, 255 in BGR)

    # ---- Save Outputs ----

    # 1. Binary mask
    cv2.imwrite(output_folder + "mask_" + frame_name, final_mask)

    # 2. Marked frame with contours
    cv2.imwrite(output_folder + "marked_" + frame_name, marked)

    # 3. Side-by-side comparison: Enhanced | Mask | Marked
    enhanced_bgr = cv2.imread(input_folder + frame_name)
    if enhanced_bgr is None:
        enhanced_bgr = img

    mask_bgr   = cv2.cvtColor(final_mask, cv2.COLOR_GRAY2BGR)
    comparison = np.hstack((enhanced_bgr, mask_bgr, marked))
    cv2.imwrite(output_folder + "comparison_" + frame_name, comparison)

print("Done. All outputs saved in:", output_folder)


# Step 7: Calculate Evaluation Metrics

print("\n--- Evaluation Metrics (Frames with Ground Truth) ---")
print("Frame                  Accuracy  Precision  Recall    F1        IoU       Dice")
print("-" * 90)

results = []

for frame_name in frames_with_gt:

    img    = cv2.imread(raveling_input + frame_name)
    gt_img = cv2.imread(gt_folder + frame_name, cv2.IMREAD_GRAYSCALE)

    if img is None or gt_img is None:
        print("Could not read:", frame_name, "- Skipping.")
        continue

    # Run segmentation
    final_mask, _, _, _ = segment_raveling_frame(img)

    # Resize ground truth to match mask size just in case
    gt_img = cv2.resize(gt_img, (final_mask.shape[1], final_mask.shape[0]))

    # Convert both to 0 and 1
    _, gt_bin   = cv2.threshold(gt_img,     127, 1, cv2.THRESH_BINARY)
    _, pred_bin = cv2.threshold(final_mask, 127, 1, cv2.THRESH_BINARY)

    # Count TP, FP, FN, TN
    TP = int(np.sum((pred_bin == 1) & (gt_bin == 1)))
    FP = int(np.sum((pred_bin == 1) & (gt_bin == 0)))
    FN = int(np.sum((pred_bin == 0) & (gt_bin == 1)))
    TN = int(np.sum((pred_bin == 0) & (gt_bin == 0)))

    total = TP + FP + FN + TN

    accuracy = (TP + TN) / total

    if (TP + FP) == 0:
        precision = 0.0
    else:
        precision = TP / (TP + FP)

    if (TP + FN) == 0:
        recall = 0.0
    else:
        recall = TP / (TP + FN)

    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    if (TP + FP + FN) == 0:
        iou = 0.0
    else:
        iou = TP / (TP + FP + FN)

    if (2 * TP + FP + FN) == 0:
        dice = 0.0
    else:
        dice = (2 * TP) / (2 * TP + FP + FN)

    results.append([frame_name, accuracy, precision, recall, f1, iou, dice])

    print(frame_name, "\t",
          round(accuracy,  4), "\t",
          round(precision, 4), "\t",
          round(recall,    4), "\t",
          round(f1,        4), "\t",
          round(iou,       4), "\t",
          round(dice,      4))


# Step 8: Calculate and Print Averages

if len(results) > 0:

    total_acc = 0
    total_pre = 0
    total_rec = 0
    total_f1  = 0
    total_iou = 0
    total_dic = 0

    for r in results:
        total_acc = total_acc + r[1]
        total_pre = total_pre + r[2]
        total_rec = total_rec + r[3]
        total_f1  = total_f1  + r[4]
        total_iou = total_iou + r[5]
        total_dic = total_dic + r[6]

    count = len(results)

    avg_acc = total_acc / count
    avg_pre = total_pre / count
    avg_rec = total_rec / count
    avg_f1  = total_f1  / count
    avg_iou = total_iou / count
    avg_dic = total_dic / count

    print("-" * 90)
    print("AVERAGE", "\t\t\t",
          round(avg_acc, 4), "\t",
          round(avg_pre, 4), "\t",
          round(avg_rec, 4), "\t",
          round(avg_f1,  4), "\t",
          round(avg_iou, 4), "\t",
          round(avg_dic, 4))


# Step 9: Save Metrics to Text File

    result_file = open(output_folder + "raveling_metrics.txt", "w")
    result_file.write("Raveling Segmentation - Evaluation Results (Fixed Version)\n")
    result_file.write("Member 4 - ICT2403 Graphics and Image Processing\n")
    result_file.write("Degradation Type: Raveling\n")
    result_file.write("Threshold Value Used: 30\n")
    result_file.write("ROI: Left 12% and Right 20% and Top 15% excluded\n")
    result_file.write("Total frames tested: " + str(len(frame_files)) + "\n")
    result_file.write("Frames with ground truth: " + str(len(results)) + "\n")
    result_file.write("-" * 90 + "\n")
    result_file.write("Frame                  Accuracy  Precision  Recall    F1        IoU       Dice\n")
    result_file.write("-" * 90 + "\n")

    for r in results:
        result_file.write(r[0] + "\t" +
                          str(round(r[1], 4)) + "\t" +
                          str(round(r[2], 4)) + "\t" +
                          str(round(r[3], 4)) + "\t" +
                          str(round(r[4], 4)) + "\t" +
                          str(round(r[5], 4)) + "\t" +
                          str(round(r[6], 4)) + "\n")

    result_file.write("-" * 90 + "\n")
    result_file.write("AVERAGE\t\t\t" +
                      str(round(avg_acc, 4)) + "\t" +
                      str(round(avg_pre, 4)) + "\t" +
                      str(round(avg_rec, 4)) + "\t" +
                      str(round(avg_f1,  4)) + "\t" +
                      str(round(avg_iou, 4)) + "\t" +
                      str(round(avg_dic, 4)) + "\n")
    result_file.close()

    print("\nMetrics saved to:", output_folder + "raveling_metrics.txt")

else:
    print("\nNo ground truth masks found. Add masks to:", gt_folder)


# Step 10: Save Clean Report Example Image

example_name = frame_files[0]
img_enhanced = cv2.imread(input_folder + example_name)
img_marked   = cv2.imread(output_folder + "marked_" + example_name)
img_mask     = cv2.imread(output_folder + "mask_"   + example_name, cv2.IMREAD_GRAYSCALE)

if img_enhanced is None:
    img_enhanced = cv2.imread(raveling_input + example_name)

if img_enhanced is not None and img_marked is not None and img_mask is not None:

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(img_enhanced, cv2.COLOR_BGR2RGB))
    plt.title("Enhanced Frame (Input)")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(img_mask, cmap='gray')
    plt.title("Segmentation Mask")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(img_marked, cv2.COLOR_BGR2RGB))
    plt.title("Marked Output - Raveling")
    plt.axis('off')

    plt.suptitle("Raveling Segmentation - Final Result (Fixed Version)")
    plt.tight_layout()
    plt.savefig(output_folder + "report_example.jpg")
    plt.show()

    print("Report example saved:", output_folder + "report_example.jpg")
